# Kaleidoscope Quickstart

This notebook builds every preview clip in memory. It requires VapourSynth R77 and Kaleidoscope to be installed in the active Python 3.12 kernel.

## Generate RGB24 clips

The clips share one timeline and resolution, so they can be viewed in every comparison mode without external files or source plugins.

In [ ]:
import vapoursynth as vs
from kaleidoscope import preview

core = vs.core
source = core.std.BlankClip(
    width=640,
    height=360,
    format=vs.RGB24,
    length=120,
    fpsnum=24000,
    fpsden=1001,
    color=[220, 40, 40],
)
filtered = core.std.BlankClip(clip=source, color=[40, 120, 220])

## Preview and compare

The mapping preserves the labels and order shown in the player. Switch among side-by-side, wipe, overlay, and difference without rerendering the synchronized pair.

In [ ]:
player = preview(
    {"Source": source, "Filtered": filtered},
    mode="side-by-side",
)
player

## Discover registered outputs

Calling `preview()` without clips snapshots registered video outputs and ignores audio outputs.

In [ ]:
if globals().get("registered_player") is not None:
    registered_player.close()
registered_player = None
for output_id, expected_clip in globals().get("registered_outputs", []):
    output = vs.get_outputs().get(output_id)
    if isinstance(output, vs.VideoOutputTuple) and output.clip is expected_clip:
        vs.clear_output(output_id)

registered_output_ids = []
for output_id in range(1024, 2048):
    if output_id not in vs.get_outputs():
        registered_output_ids.append(output_id)
        if len(registered_output_ids) == 2:
            break

if len(registered_output_ids) != 2:
    raise RuntimeError("The quickstart needs two free output IDs in 1024..2047.")

registered_outputs = [
    (registered_output_ids[0], source),
    (registered_output_ids[1], filtered),
]
try:
    for output_id, clip in registered_outputs:
        clip.set_output(output_id)
    registered_player = preview(output_ids=registered_output_ids)
except BaseException:
    for output_id, expected_clip in registered_outputs:
        output = vs.get_outputs().get(output_id)
        if isinstance(output, vs.VideoOutputTuple) and output.clip is expected_clip:
            vs.clear_output(output_id)
    registered_outputs = []
    raise
registered_player

## Inspect the automatic RGB24 fallback warning

Passing a non-RGB24 clip is supported, but Kaleidoscope displays a warning because it must prepare an RGB24 conversion for browser encoding.

In [ ]:
yuv_clip = core.std.BlankClip(
    width=640,
    height=360,
    format=vs.YUV420P8,
    length=120,
    fpsnum=24000,
    fpsden=1001,
    color=[81, 90, 240],
)
warning_player = preview(yuv_clip)
warning_player

## Cleanup

Close every player in long-lived kernels. Only the registered outputs created by this example are removed.

In [ ]:
player.close()
registered_player.close()
warning_player.close()

for output_id, expected_clip in registered_outputs:
    output = vs.get_outputs().get(output_id)
    if isinstance(output, vs.VideoOutputTuple) and output.clip is expected_clip:
        vs.clear_output(output_id)